In [15]:
# ── Cell 0: Test Path ──────────────────────────────────────────

import glob
import os

# Define the path to the traffic data CSV files
data_path = r"..\..\traffic_data\*.csv"
all_files = glob.glob(data_path)
print(os.path.abspath(data_path)) 
print(f"找到 {len(all_files)} 个文件")

e:\Study\Unitec\COMP8831_Machine_Learning\Assignment_4\code\traffic_data\*.csv
找到 47 个文件


In [14]:
# ── Cell 1: Read Dataset ──────────────────────────────────────────

import pandas as pd
import glob
import os

# Path to all CSV files using wildcard (*.csv matches all CSV files in the folder)
data_path = r"..\..\traffic_data\*.csv"

# glob.glob() finds all files matching the pattern, sorted() ensures chronological order
all_files = sorted(glob.glob(data_path))
print(f"Found {len(all_files)} files total")

dfs = []     # List to store filtered DataFrames from each file
skipped = [] # List to track old-format files that are skipped

for f in all_files:
    # Read only the first row to check column names (saves memory and time)
    df_temp = pd.read_csv(f, nrows=1, encoding='latin-1')
    
    # If 'SITE_ALIAS' column not found, this is old format (pre-2020) → skip it
    if 'SITE_ALIAS' not in df_temp.columns:
        skipped.append(os.path.basename(f))
        continue
    
    # New format confirmed → read the full file
    df_temp = pd.read_csv(f, encoding='latin-1')
    
    # Filter to keep only Site 2078 (Northbound) and 2079 (Southbound) on SH1 Green Lane East
    df_temp = df_temp[df_temp['SITE_ALIAS'].isin([2078, 2079])]
    
    dfs.append(df_temp)
    print(f"✓ {os.path.basename(f)} — rows after filtering: {len(df_temp)}")

print(f"\nSkipped (old format, pre-2020): {len(skipped)} files")

# Concatenate all monthly DataFrames into one
# ignore_index=True resets the row index to be continuous (0, 1, 2, ...)
df = pd.concat(dfs, ignore_index=True)
print(f"\nMerge complete! Total rows: {len(df)}, Total columns: {df.shape[1]}")
print(df.head())

Found 47 files total
✓ tms_2020_10.csv — rows after filtering: 35712
✓ tms_2020_11.csv — rows after filtering: 34560
✓ tms_2020_12.csv — rows after filtering: 35712
✓ tms_2021_01.csv — rows after filtering: 35712
✓ tms_2021_02.csv — rows after filtering: 32256
✓ tms_2021_03.csv — rows after filtering: 35712
✓ tms_2021_04.csv — rows after filtering: 34560
✓ tms_2021_05.csv — rows after filtering: 35712
✓ tms_2021_06.csv — rows after filtering: 34560
✓ tms_2021_07.csv — rows after filtering: 35712
✓ tms_2021_08.csv — rows after filtering: 35712
✓ tms_2021_09.csv — rows after filtering: 34560
✓ tms_2021_10.csv — rows after filtering: 35712
✓ tms_2021_11.csv — rows after filtering: 34560
✓ tms_2021_12.csv — rows after filtering: 35712
✓ tms_2022_01.csv — rows after filtering: 26496

Skipped (old format, pre-2020): 31 files

Merge complete! Total rows: 552960, Total columns: 9
         START_DATE  SITE_ALIAS    REGION_NAME SITE_REFERENCE CLASS_WEIGHT  \
0  2020-10-01 00:30        2079  02 -

In [16]:
# ── Cell 2: Basic Overview ──────────────────────────────────────────

# Convert START_DATE column from string to datetime format for time-based operations
df['START_DATE'] = pd.to_datetime(df['START_DATE'])

# Print the overall date range and total number of records
print("=== Basic Info ===")
print(f"Date range: {df['START_DATE'].min()} → {df['START_DATE'].max()}")
print(f"Total rows: {len(df)}")

# Check how many records belong to each site
# Site 2078 = SH1 Green Lane East Northbound
# Site 2079 = SH1 Green Lane East Southbound
print(f"\n=== Sites Count ===")
print(df['SITE_ALIAS'].value_counts())

# Check the ratio of Light vs Heavy vehicles
# We will aggregate both classes later, so this confirms both types are present
print(f"\n=== Class Weight ===")
print(df['CLASS_WEIGHT'].value_counts())

# Check for missing values in each column
# Any column with missing values will need to be handled before model training
print(f"\n=== Missing Values ===")
print(df.isnull().sum())

# Show basic statistics of traffic count (min, max, mean, std, quartiles)
# Helps detect outliers and understand the value range before normalisation
print(f"\n=== TRAFFIC_COUNT Stats ===")
print(df['TRAFFIC_COUNT'].describe())

=== Basic Info ===
Date range: 2020-10-01 00:00:00 → 2022-01-23 23:45:00
Total rows: 552960

=== Sites Count ===
SITE_ALIAS
2079    276480
2078    276480
Name: count, dtype: int64

=== Class Weight ===
CLASS_WEIGHT
Light    276480
Heavy    276480
Name: count, dtype: int64

=== Missing Values ===
START_DATE          0
SITE_ALIAS          0
REGION_NAME         0
SITE_REFERENCE      0
CLASS_WEIGHT        0
SITE_DESCRIPTION    0
LANE_NUMBER         0
FLOW_DIRECTION      0
TRAFFIC_COUNT       0
dtype: int64

=== TRAFFIC_COUNT Stats ===
count    552960.000000
mean        100.596566
std         129.563294
min           0.000000
25%           7.500000
50%          26.000000
75%         189.500000
max         578.000000
Name: TRAFFIC_COUNT, dtype: float64


In [18]:
# ── Cell 3: Aggregate Light + Heavy vehicles ─────────────────────────

# The raw data has separate rows for Light and Heavy vehicles at the same timestamp.
# We need to combine them to get the TOTAL vehicle count per 15-min interval.
# Strategy: group by time + site + lane + direction, then sum TRAFFIC_COUNT.

df_agg = df.groupby(
    ['START_DATE', 'SITE_ALIAS', 'LANE_NUMBER', 'FLOW_DIRECTION']
)['TRAFFIC_COUNT'].sum().reset_index()
# reset_index() converts the grouped result back into a regular DataFrame

# Rename columns for clarity after groupby
df_agg.columns = ['START_DATE', 'SITE_ALIAS', 'LANE_NUMBER', 'FLOW_DIRECTION', 'TRAFFIC_COUNT']

# Verify the aggregation halved the row count (Light + Heavy → Total)
print(f"Rows before aggregation (Light + Heavy separate): {len(df)}")
print(f"Rows after aggregation  (Total combined):         {len(df_agg)}")

print(f"\nSample data after aggregation:")
print(df_agg.head(10))

Rows before aggregation (Light + Heavy separate): 552960
Rows after aggregation  (Total combined):         276480

Sample data after aggregation:
           START_DATE  SITE_ALIAS  LANE_NUMBER  FLOW_DIRECTION  TRAFFIC_COUNT
0 2020-10-01 00:00:00        2078            1               1           75.0
1 2020-10-01 00:00:00        2078            2               1           71.0
2 2020-10-01 00:00:00        2078            3               1           27.0
3 2020-10-01 00:00:00        2079            1               2           41.0
4 2020-10-01 00:00:00        2079            2               2           67.0
5 2020-10-01 00:00:00        2079            3               2           13.0
6 2020-10-01 00:15:00        2078            1               1           67.0
7 2020-10-01 00:15:00        2078            2               1           50.0
8 2020-10-01 00:15:00        2078            3               1           12.0
9 2020-10-01 00:15:00        2079            1               2           3

In [19]:
# ── Cell 4: Aggregate by Lane ─────────────────────────────────────────

# Each site has multiple lanes. We sum all lanes to get the total flow
# per site per 15-min interval, which represents the full road capacity.

df_site = df_agg.groupby(
    ['START_DATE', 'SITE_ALIAS', 'FLOW_DIRECTION']
)['TRAFFIC_COUNT'].sum().reset_index()
# After this step, each row = one 15-min interval at one site (all lanes combined)

print(f"Rows after lane aggregation: {len(df_site)}")
print(f"\nSample data:")
print(df_site.head(10))

# Check how many unique timestamps exist per site
print(f"\nUnique timestamps per site:")
print(df_site.groupby('SITE_ALIAS')['START_DATE'].nunique())

Rows after lane aggregation: 92160

Sample data:
           START_DATE  SITE_ALIAS  FLOW_DIRECTION  TRAFFIC_COUNT
0 2020-10-01 00:00:00        2078               1          173.0
1 2020-10-01 00:00:00        2079               2          121.0
2 2020-10-01 00:15:00        2078               1          129.0
3 2020-10-01 00:15:00        2079               2          111.0
4 2020-10-01 00:30:00        2078               1          116.0
5 2020-10-01 00:30:00        2079               2           85.0
6 2020-10-01 00:45:00        2078               1          120.0
7 2020-10-01 00:45:00        2079               2           62.0
8 2020-10-01 01:00:00        2078               1          111.0
9 2020-10-01 01:00:00        2079               2           85.0

Unique timestamps per site:
SITE_ALIAS
2078    46080
2079    46080
Name: START_DATE, dtype: int64


In [20]:
# ── Cell 5: Check Time Continuity ────────────────────────────────────

# Separate the two sites for individual analysis
df_2078 = df_site[df_site['SITE_ALIAS'] == 2078].sort_values('START_DATE').reset_index(drop=True)
df_2079 = df_site[df_site['SITE_ALIAS'] == 2079].sort_values('START_DATE').reset_index(drop=True)

# Check for gaps in the time series (expected interval = 15 minutes)
def check_time_gaps(df, site_name):
    time_diff = df['START_DATE'].diff().dropna()
    expected = pd.Timedelta('15min')
    gaps = time_diff[time_diff != expected]
    print(f"\n{site_name}:")
    print(f"  Total timestamps: {len(df)}")
    print(f"  Expected interval: 15 min")
    print(f"  Gaps found: {len(gaps)}")
    if len(gaps) > 0:
        print(f"  Gap details:\n{gaps}")

check_time_gaps(df_2078, 'Site 2078 (Northbound)')
check_time_gaps(df_2079, 'Site 2079 (Southbound)')


Site 2078 (Northbound):
  Total timestamps: 46080
  Expected interval: 15 min
  Gaps found: 0

Site 2079 (Southbound):
  Total timestamps: 46080
  Expected interval: 15 min
  Gaps found: 0


In [22]:
# ── Cell 6: Outlier Detection ─────────────────────────────────────────

# We use the IQR (Interquartile Range) method to detect outliers.
# IQR is the range between the 25th percentile (Q1) and 75th percentile (Q3).
# Any value that falls too far from the middle range is considered an outlier.
#
# Visual explanation:
# |----lower_bound----|----Q1----|----median----|----Q3----|----upper_bound----|
#        outlier zone      normal data range                    outlier zone
#
# Formula:
#   lower_bound = Q1 - 1.5 * IQR  (anything below this is unusually low)
#   upper_bound = Q3 + 1.5 * IQR  (anything above this is unusually high)

def check_outliers(df, site_name):
    # Calculate Q1 (25th percentile) and Q3 (75th percentile)
    Q1 = df['TRAFFIC_COUNT'].quantile(0.25)  # 25% of values are below this
    Q3 = df['TRAFFIC_COUNT'].quantile(0.75)  # 75% of values are below this
    IQR = Q3 - Q1                            # The spread of the middle 50% of data
    
    # Define the boundaries for normal data
    lower_bound = Q1 - 1.5 * IQR  # Values below this are abnormally low
    upper_bound = Q3 + 1.5 * IQR  # Values above this are abnormally high
    
    # Find all rows where TRAFFIC_COUNT is outside the normal range
    outliers = df[(df['TRAFFIC_COUNT'] < lower_bound) | 
                  (df['TRAFFIC_COUNT'] > upper_bound)]
    
    print(f"\n{site_name}:")
    print(f"  Q1={Q1:.1f}, Q3={Q3:.1f}, IQR={IQR:.1f}")
    print(f"  Normal range: {lower_bound:.1f} ~ {upper_bound:.1f}")
    print(f"  Outliers found: {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)")
    print(f"  Max value: {df['TRAFFIC_COUNT'].max():.1f}")

check_outliers(df_2078, 'Site 2078 (Northbound)')
check_outliers(df_2079, 'Site 2079 (Southbound)')


Site 2078 (Northbound):
  Q1=195.0, Q3=1016.0, IQR=821.0
  Normal range: -1036.5 ~ 2247.5
  Outliers found: 0 (0.00%)
  Max value: 1405.0

Site 2079 (Southbound):
  Q1=153.0, Q3=1025.0, IQR=872.0
  Normal range: -1155.0 ~ 2333.0
  Outliers found: 0 (0.00%)
  Max value: 1395.0
